# Shane Waldron

## In Class Assignment 4

### Feature Engineering

In [1]:
# pip install feature-engine

In [2]:
# importing libraries
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.metrics import balanced_accuracy_score, classification_report
from feature_engine.encoding import RareLabelEncoder, CountFrequencyEncoder
from feature_engine.discretisation import EqualFrequencyDiscretiser
from feature_engine.selection import DropConstantFeatures
from sklearn.preprocessing import PolynomialFeatures

In [3]:
adult = pd.read_csv("/Users/shanewaldron/Desktop/MSBA/GSB 545/In-Class Assignments/data/adult.csv")
adult.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [4]:
# preparing the data for modeling

# convert target variable to binary
adult["income"] = adult["income"].apply(lambda x: 1 if x == ">50K" else 0)

# drop the fnlwgt variable as it is not useful for modeling
adult.drop(columns=["fnlwgt"], inplace=True)

# gender to binary
adult["gender"] = adult["gender"].apply(lambda x: 1 if x == "Male" else 0)

# replace ? with NaN
adult.replace("?", np.nan, inplace=True)

# impute missing values as average in numerical and "unknown" in categorical
for col in adult.columns:
    if adult[col].dtype == "object":
        adult[col] = adult[col].fillna("unknown")
    else:
        adult[col] = adult[col].fillna(adult[col].median())
        
adult.head()

,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,11th,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States,0
1,38,Private,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States,0
2,28,Local-gov,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States,1
3,44,Private,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States,1
4,18,unknown,Some-college,10,Never-married,unknown,Own-child,White,0,0,0,30,United-States,0


In [5]:
# Feature Preprocessing with Feature-engine

from feature_engine.encoding import RareLabelEncoder, CountFrequencyEncoder
from feature_engine.discretisation import EqualFrequencyDiscretiser
from feature_engine.selection import DropConstantFeatures
from sklearn.model_selection import train_test_split

# copy data
df_fe = adult.copy()

# split features/target
X = df_fe.drop("income", axis=1)
y = df_fe["income"]

# split data
X_train_fe, X_test_fe, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# identify variable types
cat_cols = X_train_fe.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_train_fe.select_dtypes(include=["int64", "float64"]).columns.tolist()

# collapse rare categories
rare_encoder = RareLabelEncoder(
    tol=0.01,
    variables=cat_cols
)

X_train_fe = rare_encoder.fit_transform(X_train_fe)
X_test_fe = rare_encoder.transform(X_test_fe)

# frequency encode categorical variables
freq_encoder = CountFrequencyEncoder(variables=cat_cols, encoding_method="frequency")

X_train_fe = freq_encoder.fit_transform(X_train_fe)
X_test_fe = freq_encoder.transform(X_test_fe)

# exclude problematic variables from discretization
disc_vars = [col for col in num_cols if col not in ["gender", "capital-gain", "capital-loss"]]

# discretize remaining numeric variables
disc = EqualFrequencyDiscretiser(
    q=5,
    variables=disc_vars
)

X_train_fe = disc.fit_transform(X_train_fe)
X_test_fe = disc.transform(X_test_fe)

# drop constant features if any were created
const_drop = DropConstantFeatures()

X_train_fe = const_drop.fit_transform(X_train_fe)
X_test_fe = const_drop.transform(X_test_fe)

print("Original shape:", X.shape)
print("Transformed train shape:", X_train_fe.shape)

X_train_fe.head(20)

Original shape: (48842, 13)
Transformed train shape: (39073, 13)


/opt/anaconda3/lib/python3.13/site-packages/feature_engine/encoding/rare_label.py:216: UserWarning: The number of unique categories for variable workclass is less than that indicated in n_categories. Thus, all categories will be considered frequent
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/feature_engine/encoding/rare_label.py:216: UserWarning: The number of unique categories for variable marital-status is less than that indicated in n_categories. Thus, all categories will be considered frequent
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/feature_engine/encoding/rare_label.py:216: UserWarning: The number of unique categories for variable relationship is less than that indicated in n_categories. Thus, all categories will be considered frequent
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/feature_engine/encoding/rare_label.py:216: UserWarning: The number of unique categories for variable race is less than that indicated in n_categories.

,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country
34342,4,0.693727,0.324751,0,0.330945,0.042664,0.258618,0.85609,1,0,0,0,0.898344
18559,0,0.693727,0.028613,0,0.330945,0.113122,0.031352,0.85609,0,0,0,0,0.898344
12477,1,0.693727,0.324751,0,0.456632,0.101144,0.402580,0.85609,1,0,0,1,0.064699
560,3,0.693727,0.324751,0,0.031684,0.115067,0.105239,0.09513,0,0,0,1,0.898344
3427,1,0.693727,0.164257,2,0.456632,0.125278,0.402580,0.85609,1,0,0,1,0.898344
40152,0,0.693727,0.220920,1,0.330945,0.042664,0.154890,0.85609,1,0,0,0,0.898344
23445,0,0.056919,0.220920,1,0.330945,0.057175,0.154890,0.85609,0,0,0,0,0.898344
30132,3,0.693727,0.324751,0,0.456632,0.113122,0.402580,0.85609,1,0,0,1,0.898344
44628,3,0.029458,0.324751,0,0.330945,0.115067,0.105239,0.85609,0,0,0,1,0.898344
18571,0,0.693727,0.324751,0,0.330945,0.042664,0.154890,0.85609,0,0,1719,0,0.898344


In [6]:
print("Dropped constant features:", const_drop.features_to_drop_)

Dropped constant features: []


In [7]:
rare_encoder.encoder_dict_

{'workclass': ['Private',
  'unknown',
  'Federal-gov',
  'Local-gov',
  'State-gov',
  'Self-emp-not-inc',
  'Self-emp-inc',
  'Never-worked',
  'Without-pay'],
 'education': ['HS-grad',
  'Some-college',
  'Bachelors',
  'Masters',
  'Assoc-voc',
  '11th',
  'Assoc-acdm',
  '10th',
  '7th-8th',
  'Prof-school',
  '9th',
  '12th',
  'Doctorate',
  '5th-6th'],
 'marital-status': ['Never-married',
  'Married-civ-spouse',
  'Separated',
  'Divorced',
  'Widowed',
  'Married-AF-spouse',
  'Married-spouse-absent'],
 'occupation': ['Prof-specialty',
  'Craft-repair',
  'Exec-managerial',
  'Adm-clerical',
  'Sales',
  'Other-service',
  'Machine-op-inspct',
  'unknown',
  'Transport-moving',
  'Handlers-cleaners',
  'Farming-fishing',
  'Tech-support',
  'Protective-serv'],
 'relationship': ['Not-in-family',
  'Other-relative',
  'Husband',
  'Unmarried',
  'Own-child',
  'Wife'],
 'race': ['White',
  'Black',
  'Amer-Indian-Eskimo',
  'Other',
  'Asian-Pac-Islander'],
 'native-country': ['

In [8]:
# PolynomialFeatures on selected Feature-engine columns

X_train_poly_input = X_train_fe.copy()
X_test_poly_input = X_test_fe.copy()

poly = PolynomialFeatures(
    degree=3,
    interaction_only=True,
    include_bias=False
)

# fit on training data only
X_train_poly = poly.fit_transform(X_train_poly_input)

# transform test data using the same fitted transformer
X_test_poly = poly.transform(X_test_poly_input)

# feature names
poly_feature_names = poly.get_feature_names_out(X_train_poly_input.columns)

# convert back to DataFrames
X_train_poly_df = pd.DataFrame(
    X_train_poly,
    columns=poly_feature_names,
    index=X_train_poly_input.index
)

X_test_poly_df = pd.DataFrame(
    X_test_poly,
    columns=poly_feature_names,
    index=X_test_poly_input.index
)

print("Selected FE train shape:", X_train_poly_input.shape)
print("Expanded train shape:", X_train_poly_df.shape)
print("Selected FE test shape:", X_test_poly_input.shape)
print("Expanded test shape:", X_test_poly_df.shape)

X_train_poly_df.head()

Selected FE train shape: (39073, 13)
Expanded train shape: (39073, 377)
Selected FE test shape: (9769, 13)
Expanded test shape: (9769, 377)


,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,...,gender capital-gain capital-loss,gender capital-gain hours-per-week,gender capital-gain native-country,gender capital-loss hours-per-week,gender capital-loss native-country,gender hours-per-week native-country,capital-gain capital-loss hours-per-week,capital-gain capital-loss native-country,capital-gain hours-per-week native-country,capital-loss hours-per-week native-country
34342,4.0,0.693727,0.324751,0.0,0.330945,0.042664,0.258618,0.85609,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
18559,0.0,0.693727,0.028613,0.0,0.330945,0.113122,0.031352,0.85609,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
12477,1.0,0.693727,0.324751,0.0,0.456632,0.101144,0.402580,0.85609,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.064699,0.0,0.0,0.0,0.0
560,3.0,0.693727,0.324751,0.0,0.031684,0.115067,0.105239,0.09513,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
3427,1.0,0.693727,0.164257,2.0,0.456632,0.125278,0.402580,0.85609,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.898344,0.0,0.0,0.0,0.0


### Baseline models with transformed and expanded features
I first compare the feature-engineered dataset to the larger polynomial feature set so I can see whether adding complexity actually helps.

In [9]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
spw = (y_train == 0).sum() / (y_train == 1).sum()

rf_baseline_fe = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_estimators=200,
    n_jobs=-1
)
xgb_baseline_fe = XGBClassifier(
    scale_pos_weight=spw,
    eval_metric='logloss',
    random_state=42,
    n_estimators=200
)

rf_baseline_poly = clone(rf_baseline_fe)
xgb_baseline_poly = clone(xgb_baseline_fe)

rf_baseline_fe.fit(X_train_fe, y_train)
rf_preds_fe = rf_baseline_fe.predict(X_test_fe)

xgb_baseline_fe.fit(X_train_fe, y_train)
xgb_preds_fe = xgb_baseline_fe.predict(X_test_fe)

rf_baseline_poly.fit(X_train_poly_df, y_train)
rf_preds_poly = rf_baseline_poly.predict(X_test_poly_df)

xgb_baseline_poly.fit(X_train_poly_df, y_train)
xgb_preds_poly = xgb_baseline_poly.predict(X_test_poly_df)

baseline_results = pd.DataFrame([
    ['Random Forest', 'Feature-engine set', X_train_fe.shape[1], balanced_accuracy_score(y_test, rf_preds_fe)],
    ['Random Forest', 'Polynomial set', X_train_poly_df.shape[1], balanced_accuracy_score(y_test, rf_preds_poly)],
    ['XGBoost', 'Feature-engine set', X_train_fe.shape[1], balanced_accuracy_score(y_test, xgb_preds_fe)],
    ['XGBoost', 'Polynomial set', X_train_poly_df.shape[1], balanced_accuracy_score(y_test, xgb_preds_poly)]
], columns=['model', 'dataset', 'n_features', 'test_balanced_accuracy'])

baseline_results.sort_values(['model', 'test_balanced_accuracy'], ascending=[True, False])

,model,dataset,n_features,test_balanced_accuracy
0,Random Forest,Feature-engine set,13,0.802323
1,Random Forest,Polynomial set,377,0.799764
2,XGBoost,Feature-engine set,13,0.836926
3,XGBoost,Polynomial set,377,0.827207


### Ranking and reducing the 377-feature polynomial set
To reduce the expanded feature space to 20 or fewer features, I average built-in feature importance and permutation importance rankings for each model, then keep the top 20 features for that model.

In [10]:
rf_perm_importance_poly = permutation_importance(
    rf_baseline_poly,
    X_test_poly_df,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring='balanced_accuracy'
)

xgb_perm_importance_poly = permutation_importance(
    xgb_baseline_poly,
    X_test_poly_df,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring='balanced_accuracy'
)

rf_feat_df = pd.DataFrame({
    'feature': X_train_poly_df.columns,
    'feat_importance': rf_baseline_poly.feature_importances_
})
rf_perm_df = pd.DataFrame({
    'feature': X_train_poly_df.columns,
    'perm_importance': rf_perm_importance_poly.importances_mean
})
rf_ranked = rf_feat_df.merge(rf_perm_df, on='feature')
rf_ranked['rank_feat'] = rf_ranked['feat_importance'].rank(ascending=False)
rf_ranked['rank_perm'] = rf_ranked['perm_importance'].rank(ascending=False)
rf_ranked['avg_rank'] = (rf_ranked['rank_feat'] + rf_ranked['rank_perm']) / 2
rf_ranked = rf_ranked.sort_values('avg_rank')
rf_top20_features = rf_ranked.head(20)['feature'].tolist()

xgb_feat_df = pd.DataFrame({
    'feature': X_train_poly_df.columns,
    'feat_importance': xgb_baseline_poly.feature_importances_
})
xgb_perm_df = pd.DataFrame({
    'feature': X_train_poly_df.columns,
    'perm_importance': xgb_perm_importance_poly.importances_mean
})
xgb_ranked = xgb_feat_df.merge(xgb_perm_df, on='feature')
xgb_ranked['rank_feat'] = xgb_ranked['feat_importance'].rank(ascending=False)
xgb_ranked['rank_perm'] = xgb_ranked['perm_importance'].rank(ascending=False)
xgb_ranked['avg_rank'] = (xgb_ranked['rank_feat'] + xgb_ranked['rank_perm']) / 2
xgb_ranked = xgb_ranked.sort_values('avg_rank')
xgb_top20_features = xgb_ranked.head(20)['feature'].tolist()

print('Random Forest top 20 features:')
print(rf_top20_features)
print()
print('XGBoost top 20 features:')
print(xgb_top20_features)

Random Forest top 20 features:
['marital-status', 'marital-status occupation', 'marital-status occupation hours-per-week', 'marital-status occupation native-country', 'age educational-num marital-status', 'age educational-num occupation', 'age educational-num relationship', 'educational-num marital-status relationship', 'occupation relationship gender', 'age educational-num hours-per-week', 'educational-num relationship hours-per-week', 'marital-status race', 'age marital-status occupation', 'educational-num occupation hours-per-week', 'marital-status relationship', 'educational-num occupation', 'marital-status occupation race', 'age educational-num', 'age marital-status hours-per-week', 'capital-gain']

XGBoost top 20 features:
['marital-status', 'capital-gain', 'educational-num occupation', 'age education hours-per-week', 'marital-status occupation', 'capital-loss', 'age educational-num hours-per-week', 'age educational-num', 'marital-status relationship capital-gain', 'marital-statu

In [11]:
X_train_rf_20 = X_train_poly_df[rf_top20_features].copy()
X_test_rf_20 = X_test_poly_df[rf_top20_features].copy()

X_train_xgb_20 = X_train_poly_df[xgb_top20_features].copy()
X_test_xgb_20 = X_test_poly_df[xgb_top20_features].copy()

reduction_summary = pd.DataFrame([
    ['Feature-engine set', X_train_fe.shape[1]],
    ['Polynomial set', X_train_poly_df.shape[1]],
    ['RF reduced set', X_train_rf_20.shape[1]],
    ['XGB reduced set', X_train_xgb_20.shape[1]]
], columns=['dataset', 'n_features'])

reduction_summary

,dataset,n_features
0,Feature-engine set,13
1,Polynomial set,377
2,RF reduced set,20
3,XGB reduced set,20


### Tuned models on the reduced feature sets
I now tune both models on their own reduced 20-feature datasets so I can compare whether simplification plus tuning works better than keeping all 377 expanded features.

In [12]:
rf_param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [8, 12, None],
    'min_samples_leaf': [1, 3, 5],
    'max_features': ['sqrt', 0.7]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_grid=rf_param_grid,
    scoring='balanced_accuracy',
    cv=skf,
    n_jobs=1
)
rf_grid.fit(X_train_rf_20, y_train)

best_rf_20 = rf_grid.best_estimator_
rf_tuned_preds = best_rf_20.predict(X_test_rf_20)

print('Best RF params:', rf_grid.best_params_)
print('Best RF CV balanced accuracy:', round(rf_grid.best_score_, 4))
print('RF tuned test balanced accuracy:', round(balanced_accuracy_score(y_test, rf_tuned_preds), 4))

Best RF params: {'max_depth': 8, 'max_features': 0.7, 'min_samples_leaf': 3, 'n_estimators': 200}
Best RF CV balanced accuracy: 0.8205
RF tuned test balanced accuracy: 0.8225


In [13]:
xgb_param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.03, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    XGBClassifier(scale_pos_weight=spw, eval_metric='logloss', random_state=42),
    param_grid=xgb_param_grid,
    scoring='balanced_accuracy',
    cv=skf,
    n_jobs=1
)
xgb_grid.fit(X_train_xgb_20, y_train)

best_xgb_20 = xgb_grid.best_estimator_
xgb_tuned_preds = best_xgb_20.predict(X_test_xgb_20)

print('Best XGB params:', xgb_grid.best_params_)
print('Best XGB CV balanced accuracy:', round(xgb_grid.best_score_, 4))
print('XGB tuned test balanced accuracy:', round(balanced_accuracy_score(y_test, xgb_tuned_preds), 4))

Best XGB params: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 400, 'subsample': 1.0}
Best XGB CV balanced accuracy: 0.8427
XGB tuned test balanced accuracy: 0.8397


In [14]:
model_comparison = pd.DataFrame([
    ['Random Forest', 'Feature-engine set', X_train_fe.shape[1], balanced_accuracy_score(y_test, rf_preds_fe)],
    ['Random Forest', 'Polynomial set', X_train_poly_df.shape[1], balanced_accuracy_score(y_test, rf_preds_poly)],
    ['Random Forest', 'Reduced + tuned', X_train_rf_20.shape[1], balanced_accuracy_score(y_test, rf_tuned_preds)],
    ['XGBoost', 'Feature-engine set', X_train_fe.shape[1], balanced_accuracy_score(y_test, xgb_preds_fe)],
    ['XGBoost', 'Polynomial set', X_train_poly_df.shape[1], balanced_accuracy_score(y_test, xgb_preds_poly)],
    ['XGBoost', 'Reduced + tuned', X_train_xgb_20.shape[1], balanced_accuracy_score(y_test, xgb_tuned_preds)]
], columns=['model', 'dataset', 'n_features', 'test_balanced_accuracy'])

model_comparison.sort_values(['model', 'test_balanced_accuracy'], ascending=[True, False])

,model,dataset,n_features,test_balanced_accuracy
2,Random Forest,Reduced + tuned,20,0.822518
0,Random Forest,Feature-engine set,13,0.802323
1,Random Forest,Polynomial set,377,0.799764
5,XGBoost,Reduced + tuned,20,0.839692
3,XGBoost,Feature-engine set,13,0.836926
4,XGBoost,Polynomial set,377,0.827207


### Stacking with a meta learner using out-of-fold predictions
The stacked model uses out-of-fold predicted probabilities from the tuned base models, then trains a logistic regression meta learner on those OOF predictions.

In [15]:
rf_oof = np.zeros(len(X_train_rf_20))
xgb_oof = np.zeros(len(X_train_xgb_20))

for train_idx, val_idx in skf.split(X_train_rf_20, y_train):
    rf_fold = clone(best_rf_20)
    xgb_fold = clone(best_xgb_20)

    rf_fold.fit(X_train_rf_20.iloc[train_idx], y_train.iloc[train_idx])
    xgb_fold.fit(X_train_xgb_20.iloc[train_idx], y_train.iloc[train_idx])

    rf_oof[val_idx] = rf_fold.predict_proba(X_train_rf_20.iloc[val_idx])[:, 1]
    xgb_oof[val_idx] = xgb_fold.predict_proba(X_train_xgb_20.iloc[val_idx])[:, 1]

X_meta_train = np.column_stack([rf_oof, xgb_oof])
meta_model = LogisticRegression(max_iter=1000)
meta_model.fit(X_meta_train, y_train)

best_rf_20.fit(X_train_rf_20, y_train)
best_xgb_20.fit(X_train_xgb_20, y_train)

rf_test_probs = best_rf_20.predict_proba(X_test_rf_20)[:, 1]
xgb_test_probs = best_xgb_20.predict_proba(X_test_xgb_20)[:, 1]

X_meta_test = np.column_stack([rf_test_probs, xgb_test_probs])
stacked_preds = meta_model.predict(X_meta_test)

print('Stacked model balanced accuracy:', round(balanced_accuracy_score(y_test, stacked_preds), 4))
print('Meta learner coefficients:', meta_model.coef_)

Stacked model balanced accuracy: 0.8158
Meta learner coefficients: [[-0.08149404  6.29122466]]


In [16]:
stacking_comparison = pd.DataFrame([
    ['Random Forest tuned', balanced_accuracy_score(y_test, rf_tuned_preds)],
    ['XGBoost tuned', balanced_accuracy_score(y_test, xgb_tuned_preds)],
    ['Stacked model', balanced_accuracy_score(y_test, stacked_preds)]
], columns=['model', 'test_balanced_accuracy'])

stacking_comparison.sort_values('test_balanced_accuracy', ascending=False)

,model,test_balanced_accuracy
1,XGBoost tuned,0.839692
0,Random Forest tuned,0.822518
2,Stacked model,0.815757


### Evaluation

Reducing the number of features made the model easier to understand, and in my results the full 377-feature polynomial set did not clearly perform better than the smaller feature sets. The features that showed up most often as important were marital-status, educational-num, age, occupation, hours-per-week, and some interactions between them, which suggests that a smaller group of personal and work-related features explains most of the prediction. I removed many of the more complex interaction terms because they seemed repetitive and did not improve balanced accuracy enough to justify the extra complexity. The reduced top-20 feature sets worked especially well for XGBoost, and tuning showed that changing model settings could make a real difference. Stacking was also useful to test because it combined the predictions from two different models using out-of-fold probabilities. Overall, this assignment showed me that adding more engineered features does not always help, and that reducing features can make the model simpler and still keep strong performance.